<a href="https://colab.research.google.com/github/SunnyChoudhary850/FlyRank/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip -q install duckdb huggingface_hub scikit-learn

In [2]:
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_content': f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':  f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
}

## Method Choice and Why

I'm using **Random Forest**, from this week's menu. My w04 baseline rule
(staleness-based) got a MIXED verdict — staleness alone doesn't cleanly
predict decline (0.689/0.541/0.647 decline rate across buckets, not a
clean trend). This suggests the real pattern needs multiple signals
combined, not one rule — exactly what Random Forest is good at, since it
can combine several imperfect signals (staleness, volume, engagement)
into one score without me hand-specifying how they interact.

I'm comparing against Logistic Regression as a simpler alternative, and
against my actual w04 baseline rule, using the same client-holdout split
and the same metric (Precision@50), so the comparison is fair.

In [3]:
features_and_label = con.sql(f"""
    WITH march_features AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS total_impressions_month,
               AVG(gsc_avg_position) AS avg_position_month,
               COUNT(DISTINCT report_date) AS days_with_activity,
               SUM(gsc_clicks) AS total_clicks_month,
               SUM(ga4_sessions) AS total_sessions_month
        FROM {TABLES['fact_daily']}
        WHERE month = '2026-03' AND gsc_data_available IS TRUE
        GROUP BY client_hash_id, content_hash_id
    ),
    april_perf AS (
        SELECT content_hash_id, SUM(gsc_impressions) AS impressions_next_month
        FROM {TABLES['fact_daily']}
        WHERE month = '2026-04' AND gsc_data_available IS TRUE
        GROUP BY content_hash_id
    ),
    staleness AS (
        SELECT content_hash_id, CURRENT_DATE - content_updated_date AS days_since_update
        FROM {TABLES['dim_content']}
    )
    SELECT f.*, a.impressions_next_month, s.days_since_update
    FROM march_features f
    JOIN april_perf a ON f.content_hash_id = a.content_hash_id
    JOIN staleness s ON f.content_hash_id = s.content_hash_id
""").df()

features_and_label['declining'] = (
    features_and_label['impressions_next_month'] < features_and_label['total_impressions_month']
).astype(int)
features_and_label = features_and_label.dropna()
features_and_label.shape

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(113853, 10)

## Split Design

I'm splitting by `client_hash_id`, not randomly — same reasoning as w03:
a random split could let the same client appear in both train and test,
letting the model learn client-specific quirks instead of real patterns.
This tests whether the model generalizes to genuinely new clients.

In [4]:
from sklearn.model_selection import GroupShuffleSplit

feature_cols = ['total_impressions_month', 'avg_position_month', 'days_with_activity',
                 'total_clicks_month', 'total_sessions_month', 'days_since_update']
X = features_and_label[feature_cols]
y = features_and_label['declining']
groups = features_and_label['client_hash_id']

splitter = GroupShuffleSplit(test_size=0.3, n_splits=1, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
print("Train rows:", len(X_train), "| Test rows:", len(X_test))

Train rows: 89753 | Test rows: 24100


In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
import numpy as np

def precision_at_k(scores, y_true, k=50):
    top_k_idx = np.argsort(scores)[-k:]
    return y_true.iloc[top_k_idx].mean()

scaler = StandardScaler().fit(X_train)

# Logistic Regression
lr = LogisticRegression(max_iter=1000).fit(scaler.transform(X_train), y_train)
lr_scores = lr.predict_proba(scaler.transform(X_test))[:, 1]
lr_p50 = precision_at_k(lr_scores, y_test.reset_index(drop=True), 50)

# Random Forest
rf = RandomForestClassifier(n_estimators=200, random_state=42).fit(X_train, y_train)
rf_scores = rf.predict_proba(X_test)[:, 1]
rf_p50 = precision_at_k(rf_scores, y_test.reset_index(drop=True), 50)

# Baseline rule from w04: pure staleness score (higher days_since_update = higher risk)
baseline_scores = X_test['days_since_update'].values
baseline_p50 = precision_at_k(baseline_scores, y_test.reset_index(drop=True), 50)

import pandas as pd
results = pd.DataFrame({
    'Model': ['w04_baseline_rule (staleness only)', 'logistic_regression', 'random_forest'],
    'Precision@50': [baseline_p50, lr_p50, rf_p50]
})
results

,Model,Precision@50
0,w04_baseline_rule (staleness only),0.56
1,logistic_regression,0.86
2,random_forest,0.74


##  Train + Compare vs My Baseline

| Model | Precision@50 |
|---|---|
| w04 baseline rule (staleness only) | 0.56 |
| Logistic Regression | 0.86 |
| Random Forest | 0.74 |

Both real models clearly beat my w04 baseline (0.56), confirming that
combining multiple signals genuinely helps over a single staleness rule.

**Honest surprise:** Logistic Regression (0.86) outperformed Random Forest
(0.74), which contradicts my Section 1 reasoning that Random Forest would
handle combined signals better. With only 6 fairly simple, mostly linear
features (impressions, position, clicks, sessions, days), there may not
be enough non-linear structure in this feature set for Random Forest's
extra complexity to pay off -- a simpler model doing the same job better
is exactly the kind of result this assignment says not to ignore
("does not reward complexity alone"). Random Forest is not automatically
the better choice just because it's more sophisticated.Train + Compare vs My Baseline

[Fill in once you see the real table above — state the actual numbers
and whether Random Forest actually beat both Logistic Regression and the
w04 staleness-only baseline]

In [6]:
importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
importances

,0
avg_position_month,0.345569
total_impressions_month,0.308311
days_with_activity,0.120310
total_sessions_month,0.082076
days_since_update,0.079654
total_clicks_month,0.064080


## Errors and Interpretation

Random Forest's top features: `avg_position_month` (0.346) and
`total_impressions_month` (0.308) dominate, together explaining over 65%
of the model's decisions. `days_since_update` (staleness) ranks lowest at
just 0.080 -- confirming what my w04 signal check already found (MIXED
verdict): staleness alone is a weak signal. The model is mostly learning
from visibility signals (position, impressions), not content age.

This means my original w04 baseline rule was built on close to the
*least* important available signal -- which explains why it only reached
0.56 while models using the full feature set reached 0.74-0.86. A better
baseline next time would have leaned on position or impressions instead
of staleness.

**What the errors likely look like:** since Logistic Regression
outperforms here, the "declining" pattern in this data is probably closer
to a straightforward, mostly linear relationship (fewer impressions +
worse position → likely declining) rather than a complex interaction
Random Forest's extra flexibility could exploit. Random Forest's added
complexity may be overfitting slightly on the training clients rather
than finding a genuinely better pattern.

## Self-Check

- [x] Compared against baseline on the same split and metric (client-holdout, Precision@50)
- [x] Used a valid grouped split, not a random one
- [x] Explained method choice, and honestly reported when the result contradicted the original reasoning (Logistic Regression beat Random Forest)
- [x] Reported Precision@50 for baseline, Logistic Regression, and Random Forest
- [x] Interpreted feature importances, and connected them back to my w04 finding that staleness was a weak signal
- [x] Did not reward complexity alone -- reported the simpler model winning